### Cart-pole with custom static controller

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/cartpole_static_controller.ipynb)


**Importing Librairies**

This page uses the toolbox *minilink*.


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
import numpy as np


# Defining the dynamics

Here we load a already defined class from the library including the dynamic equations of the cart-pole, which is a function of the form:

$\dot{x} = f(x,u)$


In [ ]:
##############
# System
##############

from minilink import CartPole, Controller

sys  = CartPole()


The dynamical equations $\dot{x} = f(x,u)$ can be represented graphically by a vector field shown here:


In [ ]:
sys.plot_phase_plane(x_axis=0, y_axis=2) # Graphical illustration of the dynamic behavior in the phase plane
sys.plot_phase_plane(x_axis=1, y_axis=3)


Since the system has 4 states, the dynamics is actually a 4d vector field. Here we ploted 2 sub-planes of this higher dimension space.


### Showing the robot natural behavior with no controllers

Here we run a simulation of the system with no controllers:


In [ ]:
# robot initial states [ joint 1 angle (rad),  joint 2 angle (rad), joint 1 velocity (rad/sec),  joint 1 velocity (rad/sec)]
sys.x0 = np.array([ 0.1, 0.1, 0.0, 0.0])

# run the simulation
sys.compute_trajectory( tf = 8 )

# Animate and display the simulation
sys.animate()


#Defining your control law


In [ ]:
class CustomController( Controller ) :

    feedback_profile = "output"

    ############################
    def __init__( self  ):
        """ """

        super().__init__()

        # Label
        self.name = 'Custom Cart-pole Controller'

        self.add_input_port("y", dim=4)
        self.add_input_port("r", dim=1, nominal_value=0.0)
        self.add_output_port("u", dim=1, function=self.ctl, dependencies=("y",))

        # Linear Gain Matric


        # TODO: Calculer des gains avec LQR!!!

        self.K = np.array([ -0.5,  25, -1.0,  5.0 ])


    #############################
    def ctl( self , x , u , t = 0, params=None ):

        y = self.get_port_values_from_u(u, "y")

        x_err = y - np.array([ 0.0, np.pi, 0.0, 0.0 ])

        u_cmd = - self.K @ x_err

        u_cmd = np.clip(u_cmd,-100.0,100.0)

        return np.array([u_cmd])

ctl = CustomController()


## Simulation

Try to play with the initial conditions:


In [ ]:
cl_sys = ctl @ sys

cl_sys.plot_diagram()

In [ ]:
sys.x0[0] = 0.5
sys.x0[1] = 2.5

cl_sys.compute_trajectory( tf = 10.0 )

In [ ]:
cl_sys.plot_trajectory()

In [ ]:
# Animate and display the simulation
cl_sys.animate()
